# FlowState: SSM и инвариантность к масштабу времени

Этот notebook содержит код из главы книги "Нейросети для прогнозирования временных рядов".

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/privettoha/neural-forecast-book/blob/main/notebooks/25_flowstate.ipynb)

**Примечание:** FlowState от IBM Research доступен для исследовательских целей.

## Установка зависимостей

In [ ]:
# FlowState требует специфичных зависимостей
!pip install -q torch pandas numpy matplotlib
# Для установки FlowState следуйте инструкциям IBM Research

## Подготовка данных

In [ ]:
import torch
import numpy as np
import pandas as pd

# Генерируем данные с суточной сезонностью
np.random.seed(42)
t = np.arange(168)  # 7 дней почасовых данных

base_series = (
    100 +                           # базовый уровень
    20 * np.sin(2 * np.pi * t / 24) +  # суточная сезонность
    0.5 * t +                       # тренд
    np.random.normal(0, 5, len(t))  # шум
)

print(f"Длина ряда: {len(base_series)}")
print(f"Min: {base_series.min():.2f}, Max: {base_series.max():.2f}")

## Вычисление scale_factor

In [ ]:
def calculate_scale_factor(frequency: str, seasonality_type: str = 'auto') -> float:
    """
    Вычисляет scale_factor для FlowState.
    
    Args:
        frequency: частота данных ('15min', '30min', 'H', 'D', 'W', 'M')
        seasonality_type: тип сезонности ('daily', 'weekly', 'yearly', 'auto')
    
    Returns:
        scale_factor для FlowState
    """
    BASE_SEASONALITY = 24  # базовая сезонность модели
    
    # Определяем сезонность данных
    seasonality_map = {
        # (frequency, seasonality_type): N
        ('15min', 'daily'): 96,    # 24*4 = 96 точек в сутках
        ('30min', 'daily'): 48,    # 24*2 = 48
        ('H', 'daily'): 24,        # 24 часа
        ('D', 'weekly'): 7,        # 7 дней в неделе
        ('D', 'yearly'): 365,      # 365 дней в году
        ('W', 'yearly'): 52,       # 52 недели
        ('M', 'yearly'): 12,       # 12 месяцев
    }
    
    if seasonality_type == 'auto':
        # Эвристика: для внутридневных — суточная, иначе — недельная/годовая
        if frequency in ['15min', '30min', 'H']:
            seasonality_type = 'daily'
        elif frequency == 'D':
            seasonality_type = 'weekly'
        else:
            seasonality_type = 'yearly'
    
    key = (frequency, seasonality_type)
    if key not in seasonality_map:
        raise ValueError(f"Unknown combination: {key}")
    
    N = seasonality_map[key]
    scale_factor = BASE_SEASONALITY / N
    
    return scale_factor

# Примеры
print("Примеры scale_factor:")
print(f"15-min data: {calculate_scale_factor('15min'):.2f}")
print(f"Hourly data: {calculate_scale_factor('H'):.2f}")
print(f"Daily data: {calculate_scale_factor('D'):.2f}")
print(f"Monthly data: {calculate_scale_factor('M'):.2f}")

## FlowState: прогнозирование (концептуальный код)

**Примечание:** Полная работа с FlowState требует установки из официального репозитория IBM.

In [ ]:
# Концептуальный код работы с FlowState
# В реальности требуется установка tsfm_public от IBM

'''
from tsfm_public import FlowStateForPrediction

# Загрузка модели
model = FlowStateForPrediction.from_pretrained("ibm-research/flowstate")
model.to('cuda')

# Подготовка данных
# FlowState ожидает формат (context_length, batch, channels)
context_length = 2048
batch_size = 32
n_channels = 1

time_series = torch.randn(context_length, batch_size, n_channels).to('cuda')

# Прогнозирование
forecast = model(
    time_series,
    scale_factor=1.0,       # для часовых данных с суточной сезонностью
    prediction_length=96,   # прогноз на 96 точек
    batch_first=False       # важно: context первое измерение
)

# Результат
predictions = forecast.prediction_outputs
# (batch, quantiles, horizon, channels)
# quantiles: 9 штук [0.1, 0.2, ..., 0.9]

median = predictions[:, 4, :, 0]  # медиана (0.5 квантиль)
'''

print("FlowState требует установки из репозитория IBM Research")

## Демонстрация: данные разной частоты

In [ ]:
import matplotlib.pyplot as plt

# Исходные часовые данные
hourly_data = base_series

# Преобразуем в разные частоты
# 15-минутные (интерполяция)
minute_15_data = np.interp(
    np.linspace(0, len(hourly_data)-1, len(hourly_data)*4),
    np.arange(len(hourly_data)),
    hourly_data
)

# Дневные (агрегация)
daily_data = hourly_data.reshape(-1, 24).mean(axis=1)

# Визуализация
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# 15-минутные
axes[0].plot(minute_15_data, color='blue')
axes[0].set_title(f'15-минутные данные (scale_factor = 0.25, N = {len(minute_15_data)})')
axes[0].grid(True, alpha=0.3)

# Часовые
axes[1].plot(hourly_data, color='green')
axes[1].set_title(f'Часовые данные (scale_factor = 1.0, N = {len(hourly_data)})')
axes[1].grid(True, alpha=0.3)

# Дневные
axes[2].plot(daily_data, color='red', marker='o')
axes[2].set_title(f'Дневные данные (scale_factor = 3.43, N = {len(daily_data)})')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Один паттерн — разная частота дискретизации', fontsize=14)
plt.tight_layout()
plt.show()

## Визуализация прогноза (шаблон)

In [ ]:
def plot_flowstate_forecast(history, quantiles, median, title=''):
    """
    Визуализация прогноза FlowState с квантилями.
    
    Args:
        history: исторические значения (1D array)
        quantiles: квантили прогноза (9, horizon)
        median: медиана прогноза (horizon,)
    """
    fig, ax = plt.subplots(figsize=(12, 5))
    
    # История
    ax.plot(range(len(history)), history, 
            color='black', label='История', linewidth=1.5)
    
    # Прогноз
    forecast_start = len(history)
    forecast_range = range(forecast_start, forecast_start + len(median))
    
    # Медиана
    ax.plot(forecast_range, median, 
            color='blue', label='Медиана', linewidth=2)
    
    # 80% интервал (p10-p90)
    ax.fill_between(
        forecast_range,
        quantiles[0],  # p10
        quantiles[8],  # p90
        alpha=0.2, color='blue', label='80% интервал'
    )
    
    # 50% интервал (p25-p75)
    ax.fill_between(
        forecast_range,
        quantiles[2],  # p30
        quantiles[6],  # p70
        alpha=0.4, color='blue', label='50% интервал'
    )
    
    ax.axvline(x=forecast_start, color='gray', linestyle='--', alpha=0.5)
    ax.legend()
    ax.set_title(title)
    ax.set_xlabel('Время')
    ax.set_ylabel('Значение')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Демонстрация с синтетическими данными
horizon = 48
np.random.seed(42)

# Имитация прогноза
sample_median = base_series[-1] + np.cumsum(np.random.randn(horizon) * 1.5)
sample_median += 20 * np.sin(2 * np.pi * np.arange(horizon) / 24)  # сезонность

# Генерируем квантили вокруг медианы
sample_quantiles = np.array([
    sample_median - 20,  # p10
    sample_median - 15,  # p20
    sample_median - 10,  # p30
    sample_median - 5,   # p40
    sample_median,       # p50
    sample_median + 5,   # p60
    sample_median + 10,  # p70
    sample_median + 15,  # p80
    sample_median + 20,  # p90
])

plot_flowstate_forecast(base_series, sample_quantiles, sample_median, 
                        title='FlowState: пример прогноза с квантилями')